# Aula 6 — Séries Temporais com Prophet e TimesFM

Na aula passada, a régua (copiar a semana anterior) errava R\\$ 63,78 por
dia, e a regressão com calendário, montada à mão, errava R\\$ 44,11. Hoje
vamos testar dois atalhos: o **Prophet**, que monta a regressão com
calendário por você, e o **TimesFM**, um modelo pré-treinado que prevê a
série sem treinar nada.

**Como usar este notebook:** rode as células de cima para baixo. O
professor roda e comenta a maior parte delas. Nas células marcadas com
**✏️ Sua vez**, quem digita é você: apague a linha `# SEU CODIGO AQUI` e
escreva o código, seguindo a dica logo acima. A célula de baixo confere
e mostra ✅ se deu certo, ou ❌ com uma pista.

### Preparando o ambiente

Esta célula confere se o Prophet e uma versão recente do `transformers`
(a biblioteca que traz o TimesFM) estão instalados, e instala o que
faltar. Rode ela primeiro e espere terminar: pode levar um minuto.

In [ ]:
!pip install -q prophet "transformers>=5.3"

In [ ]:
import logging
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Esconde avisos técnicos que só poluem a tela
warnings.filterwarnings("ignore")
print("Tudo pronto.")

## 1. Os dados e o formato que o Prophet exige

O Prophet só aceita duas colunas, com estes nomes exatos:

| Coluna | O que é |
|---|---|
| `ds` | a data |
| `y` | o valor que você quer prever |

In [ ]:
# Endereço dos dados desta aula no GitHub
URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/vendas_cafeteria.csv"

dados = pd.read_csv(URL_DADOS, parse_dates=["data"])
dados = dados.sort_values("data").reset_index(drop=True)

# O Prophet exige os nomes ds e y: renomear é sempre o primeiro passo
serie = dados.rename(columns={"data": "ds", "vendas": "y"})[["ds", "y"]]
serie.head()

O mesmo corte no tempo da aula passada: os últimos 28 dias ficam
escondidos.

In [ ]:
DIAS_DE_TESTE = 28

treino = serie.iloc[:-DIAS_DE_TESTE]
teste = serie.iloc[-DIAS_DE_TESTE:]
reais = teste["y"].to_numpy()

print(f"Treino: {len(treino)} dias, até {treino['ds'].max().date()}")
print(f"Teste:  {len(teste)} dias, a partir de {teste['ds'].min().date()}")

As mesmas métricas da Aula 5, numa função que imprime tudo numa linha.

In [ ]:
def calcular_metricas(reais, previstos):
    # Três jeitos de resumir o mesmo erro: em reais, em reais com punição
    # extra para erro grande, e em porcentagem
    erro = np.asarray(reais) - np.asarray(previstos)
    mae = np.mean(np.abs(erro))
    rmse = np.sqrt(np.mean(erro ** 2))
    mape = np.mean(np.abs(erro / np.asarray(reais))) * 100
    return mae, rmse, mape


def mostrar(nome, previstos):
    # Imprime as três métricas de uma previsão dos 28 dias de teste
    mae, rmse, mape = calcular_metricas(reais, previstos)
    print(f"{nome:<32} MAE R$ {mae:7.2f} | RMSE R$ {rmse:7.2f} | MAPE {mape:5.2f}%")

### ✏️ Sua vez: a régua

Monte de novo a sazonal ingênua: repita a última semana do treino 4
vezes, e guarde em `sazonal`. Depois mostre as métricas dela.

**Dica:** é igual à Aula 5. A última semana é
`treino["y"].iloc[-7:].to_numpy()`, e o `np.tile(algo, 4)` repete
`algo` 4 vezes. Para as métricas, use `mostrar("Sazonal ingênua", sazonal)`.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
mae_regua, _, _ = calcular_metricas(reais, sazonal)
if abs(mae_regua - 63.78) < 0.01:
    print(f"✅ A régua está de pé: R$ {mae_regua:.2f}.")
else:
    print("❌ Confira se você usou os 7 últimos dias do treino e repetiu 4 vezes.")

## 2. O Prophet

O modelo do Prophet é a soma das peças que você já conhece:

$$y(t) = g(t) + s(t) + h(t) + \varepsilon_t$$

Onde $g(t)$ é a tendência, $s(t)$ são as sazonalidades, $h(t)$ são os
feriados e $\varepsilon_t$ é o que sobra. Treinar e prever são três
linhas.

In [ ]:
from prophet import Prophet

# Esconde as mensagens de progresso do otimizador que o Prophet usa por dentro
logging.getLogger("cmdstanpy").disabled = True

modelo = Prophet()
modelo.fit(treino)

futuro = modelo.make_future_dataframe(periods=DIAS_DE_TESTE)
previsao = modelo.predict(futuro)

# A saída tem uma linha por dia, com a previsão e as bordas do intervalo
previsao[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail()

In [ ]:
figura = modelo.plot(previsao)
plt.title("Vendas observadas e previsão do Prophet")
plt.xlabel("Data")
plt.ylabel("Vendas (R$)")
plt.show()

### ✏️ Sua vez: o Prophet bateu a régua?

A `previsao` cobre o histórico inteiro **mais** os 28 dias novos. Pegue
só as últimas 28 linhas da coluna `yhat`, guarde em `prophet_sem` e
mostre as métricas.

**Dica:** `previsao["yhat"].iloc[-28:]` pega as últimas 28 linhas. Some
`.to_numpy()` no fim para virar uma lista de números.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
mae_prophet, _, _ = calcular_metricas(reais, prophet_sem)
if len(prophet_sem) == 28 and abs(mae_prophet - 64.5) < 1.5:
    print(f"✅ R$ {mae_prophet:.2f}. A biblioteca famosa empatou com a cópia da semana passada.")
else:
    print("❌ Confira se você pegou só as últimas 28 linhas da coluna yhat.")

### As peças que o modelo estimou sozinho

Este é o gráfico que vale a reunião: cada peça do modelo, separada.

In [ ]:
figura_componentes = modelo.plot_components(previsao)
plt.show()

## 3. O que o modelo não adivinha: o calendário

O Prophet descobre tendência e sazonalidade sozinho. Feriado, não: ele
não tem como saber que 25 de dezembro é diferente. A lista de datas
especiais é conhecimento de negócio, e quem entrega é você.

In [ ]:
# A tabela de feriados: uma linha por data, com um nome de grupo
feriados = pd.DataFrame({
    "holiday": "feriado_nacional",
    "ds": dados[dados["feriado"] == 1]["data"],
})
feriados.tail()

### ✏️ Sua vez: o Prophet com o calendário

Treine um novo modelo, `modelo_feriados`, entregando a tabela de
feriados. Preveja os 28 dias e guarde as últimas 28 linhas do `yhat` em
`prophet_com`.

**Dica:** copie as três linhas de treino e previsão da seção 2 e troque
os nomes. A única diferença é na criação do modelo:
`Prophet(holidays=feriados)`. Chame a tabela de saída de
`previsao_feriados`, porque vamos usá-la de novo.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
mae_com, _, _ = calcular_metricas(reais, prophet_com)
if abs(mae_com - 43.7) < 1.5:
    print(f"✅ R$ {mae_com:.2f}. Praticamente a regressão da Aula 5 (R$ 44,11): o Prophet soma as mesmas peças.")
else:
    print("❌ Confira o holidays=feriados na criação do modelo, e as últimas 28 linhas do yhat.")

### O intervalo de incerteza

Cada dia previsto vem com `yhat_lower` e `yhat_upper`, as bordas de uma
faixa que promete cobrir 80% dos casos. Quantos dias do teste caíram
dentro dela?

In [ ]:
faixa = previsao_feriados.iloc[-DIAS_DE_TESTE:]
dentro = (reais >= faixa["yhat_lower"].to_numpy()) & (reais <= faixa["yhat_upper"].to_numpy())
largura = faixa["yhat_upper"] - faixa["yhat_lower"]

print(f"Dias dentro da faixa: {dentro.sum()} de 28 ({dentro.mean():.0%})")
print(f"Largura média da faixa: R$ {largura.mean():.2f}")

## 4. Um modelo que nunca viu esta cafeteria: o TimesFM

O **TimesFM**, do Google, é um **modelo de fundação** para séries
temporais. Ele leu uma quantidade enorme de séries (buscas no Google,
visitas à Wikipédia e muitas outras) e aprendeu a prever o próximo
pedaço de uma série, como um modelo de linguagem prevê a próxima
palavra.

Por isso ele prevê uma série nova **sem treinar nela**. Não existe
`fit`: você entrega o passado, e ele devolve o futuro.

A célula abaixo baixa o modelo (cerca de 1 GB, só na primeira vez) e
descobre se o Colab tem GPU. Sem GPU, ele roda no processador mesmo, em
poucos segundos.

In [ ]:
import torch
from transformers import TimesFm2_5ModelForPrediction

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Rodando em:", dispositivo)

timesfm = TimesFm2_5ModelForPrediction.from_pretrained("google/timesfm-2.5-200m-transformers")
timesfm = timesfm.to(dispositivo)
timesfm.eval()

total_de_pesos = sum(parametro.numel() for parametro in timesfm.parameters())
pesos_formatado = f"{total_de_pesos:,}".replace(",", ".")
print(f"O TimesFM tem {pesos_formatado} pesos, e nenhum deles vai mudar hoje.")

Para não repetir código, a função abaixo recebe o passado e devolve três
coisas para os 28 dias seguintes: a previsão, a borda de baixo (10%) e a
borda de cima (90%) da faixa.

In [ ]:
def prever_timesfm(passado):
    # O modelo recebe o passado como tensor, o formato de números do PyTorch
    entrada = torch.tensor(passado, dtype=torch.float32, device=dispositivo)
    with torch.no_grad():
        saida = timesfm(past_values=[entrada])
    previsao = saida.mean_predictions[0, :DIAS_DE_TESTE].cpu().numpy()
    # As colunas 1 a 9 são os quantis de 10% a 90%
    quantis = saida.full_predictions[0, :DIAS_DE_TESTE].cpu().numpy()
    return previsao, quantis[:, 1], quantis[:, 9]


historico = treino["y"].to_numpy()
previsao_timesfm, borda_baixo, borda_cima = prever_timesfm(historico)

mostrar("TimesFM, sem treino", previsao_timesfm)

In [ ]:
plt.figure(figsize=(11, 4))
plt.fill_between(teste["ds"], borda_baixo, borda_cima, alpha=0.2, label="Faixa de 10% a 90%")
plt.plot(teste["ds"], reais, label="O que aconteceu", linewidth=3)
plt.plot(teste["ds"], previsao_timesfm, label="TimesFM")
plt.xlabel("Data")
plt.ylabel("Vendas (R$)")
plt.legend()
plt.show()

Sem treinar nada, ele bate a régua e bate o Prophet sem feriados. Mas
repare no dia 25.

In [ ]:
erro_por_dia = reais - previsao_timesfm
dia_do_natal = 21  # 25 de dezembro é o 22º dia do teste

print(f"No Natal: previu R$ {previsao_timesfm[dia_do_natal]:.2f}, vendeu R$ {reais[dia_do_natal]:.2f}")
erro_sem_natal = np.delete(np.abs(erro_por_dia), dia_do_natal).mean()
print(f"Erro médio nos outros 27 dias: R$ {erro_sem_natal:.2f}")

Nos dias comuns, ele é o melhor de todos. O problema é um dia só: o
TimesFM não tem como saber que 25 de dezembro é feriado.

### ✏️ Sua vez: quanto passado entregar?

O TimesFM não tem pesos para treinar, mas você escolhe o **contexto**:
quantos dias de passado ele recebe. Acima usamos todos os 1.068 dias.
Preveja de novo entregando só os **últimos 365 dias**, guarde a previsão
em `previsao_365` e mostre as métricas.

**Dica:** `historico[-365:]` pega os últimos 365 valores. A função
`prever_timesfm` devolve três coisas: se só a primeira interessa,
escreva `previsao_365, _, _ = ...`.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
mae_365, _, _ = calcular_metricas(reais, previsao_365)
if abs(mae_365 - 61.7) < 2:
    print(f"✅ R$ {mae_365:.2f}: pior que com o histórico inteiro. Mais passado ajuda o modelo a ver o ciclo do ano.")
else:
    print("❌ Confira: o passado entregue é historico[-365:], os últimos 365 dias.")

## 5. Juntando os dois mundos

O TimesFM traz o padrão; o calendário é você quem traz. A ideia é somar
à previsão dele o efeito de um feriado, medido no treino:

$$\hat{y}_t = \text{TimesFM}_t + w_1 \cdot \text{feriado}_t$$

O $w_1$ é a venda média de um feriado menos a de um dia comum. Nos dias
comuns, $\text{feriado}_t$ vale 0 e nada muda.

### ✏️ Sua vez: o efeito do feriado

Calcule `efeito_feriado`: a venda média dos feriados menos a venda média
dos dias comuns, **só no período de treino** (`dados_treino`, abaixo).

**Dica:** `dados_treino.groupby("feriado")["vendas"].mean()` devolve as
duas médias. A do feriado fica na posição `[1]`, a do dia comum na `[0]`.
O resultado tem que sair negativo.

In [ ]:
dados_treino = dados.iloc[:-DIAS_DE_TESTE]

In [ ]:
# SEU CODIGO AQUI

In [ ]:
if abs(efeito_feriado - (-389.68)) < 0.1:
    print("✅ Isso: cerca de R$ 390 a menos. Na regressão da Aula 5, o peso do feriado deu parecido (−387).")
else:
    print("❌ Confira: use dados_treino, e faça feriado menos dia comum (medias[1] - medias[0]).")

### ✏️ Sua vez: o TimesFM com o calendário

Some o efeito do feriado à previsão do TimesFM, só nos dias de feriado.
Guarde em `timesfm_ajustado` e mostre as métricas.

**Dica:** a coluna de feriado do teste já está pronta em
`feriado_teste`, com 0 ou 1 em cada dia. Então a conta é
`previsao_timesfm + efeito_feriado * feriado_teste`.

In [ ]:
feriado_teste = dados["feriado"].iloc[-DIAS_DE_TESTE:].to_numpy()

In [ ]:
# SEU CODIGO AQUI

In [ ]:
mae_ajustado, _, _ = calcular_metricas(reais, timesfm_ajustado)
if abs(mae_ajustado - 41.2) < 1.5:
    print(f"✅ R$ {mae_ajustado:.2f}: o melhor resultado do módulo. O modelo trouxe o padrão, você trouxe o calendário.")
else:
    print("❌ Confira a conta: previsao_timesfm + efeito_feriado * feriado_teste.")

### A faixa do TimesFM

Além da previsão, o TimesFM devolve quantis. Entre o de 10% e o de 90%
devem cair 80% dos dias, a mesma promessa da faixa do Prophet.

In [ ]:
dentro_timesfm = (reais >= borda_baixo) & (reais <= borda_cima)
print(f"TimesFM: {dentro_timesfm.sum()} de 28 dias dentro da faixa ({dentro_timesfm.mean():.0%})")
print(f"Largura média: R$ {(borda_cima - borda_baixo).mean():.2f}")

## 6. O placar do módulo

Os números da Aula 5 entram prontos, para comparar.

In [ ]:
placar = [
    ("Sazonal ingênua (a régua)", mae_regua),
    ("Regressão com calendário (Aula 5)", 44.11),
    ("ARIMA sazonal (Aula 5)", 57.57),
    ("Prophet, sem feriados", calcular_metricas(reais, prophet_sem)[0]),
    ("Prophet, com feriados", calcular_metricas(reais, prophet_com)[0]),
    ("TimesFM, sem treino", calcular_metricas(reais, previsao_timesfm)[0]),
    ("TimesFM + feriado", calcular_metricas(reais, timesfm_ajustado)[0]),
]

nomes = []
erros = []
for nome, mae in placar:
    print(f"{nome:<36} MAE R$ {mae:6.2f}")
    nomes.append(nome)
    erros.append(mae)

plt.figure(figsize=(10, 5))
plt.barh(nomes[::-1], erros[::-1])
plt.axvline(mae_regua, color="black", linestyle="--")
plt.xlabel("MAE no teste (R$)")
plt.title("O placar do módulo (a linha tracejada é a régua)")
plt.show()

Os três melhores vieram de ferramentas bem diferentes: uma regressão,
uma biblioteca de negócio e uma rede neural pré-treinada. O que eles têm
em comum é o calendário de feriados, que nenhum deles descobriu sozinho.

### ✏️ Para pensar

A dona da cafeteria tem 40 lojas e quer uma previsão para cada uma, toda
semana. Em duas ou três frases: qual ferramenta você usaria, e o que
você faria para ela não errar os feriados? Edite esta célula (duplo
clique) e escreva sua resposta no lugar deste parágrafo.